**Azure Monitor** is the unified observability platform for Azure — it collects, analyses, and acts on telemetry from your Azure resources, applications, and on-premises infrastructure.

```
Data sources                  Azure Monitor                   Actions
─────────────                 ─────────────                   ───────
Azure resources           →   Metrics (time-series)      →   Alerts
Applications (App Insights)   Logs (Log Analytics)        →   Autoscale
VMs (agents)              →   Traces / Distributed        →   Workbooks
Containers                →   Change tracking             →   Dashboards
On-premises               →   Alerts                      →   Logic Apps / Functions
```

The two primary data stores are:
- **Azure Monitor Metrics** — lightweight numerical time-series data; near-real-time; 93-day retention
- **Log Analytics workspace** — structured log data queryable with KQL; configurable retention up to 2 years

## Azure Monitor Metrics

**Metrics** are numerical values collected at regular intervals (typically every minute) from Azure resources — CPU percentage, network bytes, request count, queue depth, etc.

### Platform metrics vs custom metrics

| Type | Source | Cost |
|---|---|---|
| **Platform metrics** | Automatically collected by Azure from all resources | Free |
| **Custom metrics** | Emitted by your application code or Azure Monitor Agent | Charged per metric series |
| **Prometheus metrics** | Scraped from AKS or Azure Monitor managed Prometheus | Charged per sample |

### Metrics Explorer

The **Metrics Explorer** in the Azure portal lets you:
- Plot any metric over a time range
- Apply aggregations: **Average**, **Min**, **Max**, **Sum**, **Count**
- Split by dimension (e.g. split VM CPU by VM name)
- Pin charts to dashboards

### Metric dimensions

Many metrics have **dimensions** — additional properties that let you filter or split the data:

```
Storage account → Transactions metric → split by ResponseType, ApiName, GeoType
App Service     → Http2xx metric      → split by Instance
```

### Retention

- Platform metrics: **93 days** in Azure Monitor Metrics store
- For longer retention, route metrics to a Log Analytics workspace or Storage Account via **Diagnostic Settings**

## Diagnostic Settings

**Diagnostic Settings** control where a resource's telemetry is sent. Every Azure resource supports diagnostic settings:

```
Azure Resource
  ├── Activity Log  (control-plane operations)
  ├── Resource Logs (data-plane operations — e.g. SQL query logs, Blob access logs)
  └── Metrics
         ↓  Diagnostic Settings routes to:
         ├── Log Analytics workspace  (query with KQL)
         ├── Storage Account          (archive, cheap long-term retention)
         ├── Event Hub                (stream to SIEM, Splunk, Datadog)
         └── Partner solution         (direct integration)
```

### Activity Log

The **Activity Log** records all **control-plane operations** on Azure resources — who created, modified, or deleted what, and when. It answers: *"Who changed this resource?"*

- Retained for **90 days** by default in Azure Monitor
- Send to a Log Analytics workspace for longer retention and KQL querying
- Categories include Administrative, Security, Service Health, Alert, Autoscale, Policy, Recommendation

### Resource Logs

**Resource Logs** (formerly Diagnostic Logs) capture **data-plane operations** — the detailed activity happening inside a resource:

| Resource | Example resource log |
|---|---|
| Azure SQL Database | SQLInsights, QueryStoreRuntimeStatistics |
| Key Vault | AuditEvent (every secret read/write) |
| App Service | AppServiceHTTPLogs, AppServiceConsoleLogs |
| Storage Account | StorageRead, StorageWrite, StorageDelete |
| AKS | kube-apiserver, kube-audit, cluster-autoscaler |

Resource logs are **not collected by default** — you must configure a Diagnostic Setting to enable them.

## Log Analytics and KQL

**Log Analytics** is the query engine and storage backend for Azure Monitor logs. You write queries in **KQL (Kusto Query Language)** — a powerful, read-only query language designed for large-scale telemetry.

### Log Analytics workspace

A **workspace** is the container for log data. Key design decisions:

| Decision | Options |
|---|---|
| **Number of workspaces** | Centralised (one workspace per organisation) vs distributed (per team/region). Centralised simplifies querying and reduces cost. |
| **Retention** | 30–730 days (2 years) interactive; up to 12 years archive tier |
| **Data cap** | Optional daily cap to prevent runaway cost |
| **Commitment tiers** | Pay-as-you-go or commitment tiers (100 GB/day and above) for discounts |

### KQL essentials

```kql
// Basic query structure: Table | operator | operator
AzureActivity
| where TimeGenerated > ago(24h)
| where OperationNameValue contains "delete"
| project TimeGenerated, Caller, ResourceGroup, OperationNameValue
| order by TimeGenerated desc
```

```kql
// Summarise — aggregate data
requests                              // Application Insights table
| where timestamp > ago(1h)
| summarize count(), avg(duration), percentile(duration, 95) by name
| order by count_ desc
```

```kql
// Join activity log with resource data
AzureActivity
| where ActivityStatusValue == "Failure"
| summarize FailureCount = count() by Caller, bin(TimeGenerated, 1h)
| where FailureCount > 5
```

```kql
// VM CPU over time
Perf
| where ObjectName == "Processor" and CounterName == "% Processor Time"
| where TimeGenerated > ago(1h)
| summarize AvgCPU = avg(CounterValue) by Computer, bin(TimeGenerated, 5m)
| render timechart
```

### Common KQL operators

| Operator | Purpose |
|---|---|
| `where` | Filter rows |
| `project` | Select/rename columns |
| `extend` | Add computed columns |
| `summarize` | Aggregate with `count()`, `avg()`, `sum()`, `percentile()`, `dcount()` |
| `bin()` | Round values into buckets (e.g. 5-minute time buckets) |
| `join` | Join two tables |
| `render` | Visualise as timechart, barchart, piechart |
| `ago()` | Relative time — `ago(1h)`, `ago(7d)` |
| `top` | Return top N rows by a column |
| `distinct` | Return unique values |

## Application Insights

**Application Insights** is the APM (Application Performance Monitoring) component of Azure Monitor. It instruments your application code to collect:

| Signal | Description |
|---|---|
| **Requests** | HTTP request rate, duration, success/failure |
| **Dependencies** | Calls to SQL, Redis, HTTP APIs — latency, failure rate |
| **Exceptions** | Unhandled exceptions with stack traces |
| **Page views** | Client-side browser telemetry |
| **Custom events** | Business events you instrument (`trackEvent`) |
| **Custom metrics** | Your application metrics (`trackMetric`) |
| **Traces** | Log messages correlated with requests |
| **Availability** | Synthetic monitoring — ping tests from global PoPs |

### Workspace-based vs classic

**Workspace-based Application Insights** (recommended) stores all data in a **Log Analytics workspace** — enabling cross-resource KQL queries that join application telemetry with infrastructure logs in the same workspace.

### Distributed tracing

Application Insights correlates telemetry across microservices using a **correlation ID** propagated in HTTP headers. The **Application Map** visualises all dependencies as a graph — making it easy to identify which service is causing failures or latency in a distributed system.

### Live Metrics

**Live Metrics** streams real-time request rate, failure rate, and server count in a one-second refresh — useful for watching deployments in progress.

### Sampling

To control costs and data volume, Application Insights supports:
- **Adaptive sampling** — automatically adjusts the sample rate to keep ingestion near a target volume
- **Fixed-rate sampling** — sample a fixed percentage of requests
- **Ingestion sampling** — applied after data arrives at the backend

## Azure Monitor Alerts

**Alerts** notify you (or trigger automated remediation) when a condition is met in your telemetry.

### Alert rule components

```
Scope (resource)  →  Condition (signal + threshold)  →  Action Group  →  Notification / Action
```

| Component | Description |
|---|---|
| **Scope** | The Azure resource(s) being monitored |
| **Condition** | Signal type + evaluation logic |
| **Severity** | 0 (Critical) → 4 (Verbose) |
| **Action group** | Set of actions to take when alert fires |

### Alert signal types

| Signal type | Source | Example |
|---|---|---|
| **Metric alert** | Azure Monitor Metrics | CPU > 80% for 5 minutes |
| **Log alert** | KQL query on Log Analytics | Error count > 10 in last 15 min |
| **Activity log alert** | Azure Activity Log | VM deleted in subscription |
| **Resource Health alert** | Azure Resource Health | VM becomes unavailable |
| **Service Health alert** | Azure Service Health | Azure outage in East US |
| **Smart Detection** | Application Insights anomaly detection | Unusual failure rate increase |

### Metric alerts — evaluation

| Setting | Description |
|---|---|
| **Aggregation type** | Average / Min / Max / Total / Count |
| **Operator** | Greater than / Less than / Equal to |
| **Threshold** | Static value or **Dynamic threshold** (ML-based baseline) |
| **Evaluation frequency** | How often the condition is checked (1–60 min) |
| **Aggregation granularity** | Time window over which the metric is aggregated |

**Dynamic thresholds** use machine learning to learn the normal pattern of a metric (including seasonality) and alert only on genuine anomalies — reducing alert fatigue from static thresholds.

### Action groups

An **action group** defines what happens when an alert fires. Multiple alert rules can share one action group:

| Action type | Description |
|---|---|
| **Email / SMS / Push / Voice** | Notify individuals or distribution lists |
| **Azure app push** | Notify via Azure mobile app |
| **Webhook** | Call any HTTPS endpoint |
| **Logic App** | Trigger a Logic App workflow |
| **Azure Function** | Trigger a Function for automated remediation |
| **Event Hub** | Send alert payload to Event Hub |
| **IT Service Management (ITSM)** | Create ticket in ServiceNow, Jira |
| **Automation Runbook** | Run an Azure Automation runbook |

### Alert processing rules

**Alert processing rules** (formerly Action Rules) allow you to:
- **Suppress** alerts during maintenance windows
- **Add action groups** to all alerts matching a scope or filter
- Apply across subscriptions without modifying individual alert rules

## Azure Monitor Agent and VM Insights

### Azure Monitor Agent (AMA)

The **Azure Monitor Agent** replaces three legacy agents (Log Analytics agent, Diagnostics extension, Dependency agent). It runs on:
- Azure VMs (Windows and Linux)
- On-premises servers (via Azure Arc)
- Virtual Machine Scale Sets

AMA is configured via **Data Collection Rules (DCRs)** — JSON-based rules that define what data to collect and where to send it:

```
Data Collection Rule
  ├── Data sources: Perf counters, Windows Event Log, Syslog, custom text logs
  ├── Destinations: Log Analytics workspace(s), Azure Monitor Metrics
  └── Associations: Applied to specific VMs or resource groups
```

### VM Insights

**VM Insights** uses the Azure Monitor Agent to provide out-of-the-box dashboards for:
- **Performance** — CPU, memory, disk, network over time
- **Map** — visualise all processes and TCP connections on a VM and the services they depend on

Enable with one click — Azure deploys the required DCRs and agent extensions automatically.

### Container Insights

**Container Insights** monitors AKS clusters — collecting pod logs, node metrics, and container performance into a Log Analytics workspace. It uses the Azure Monitor Agent deployed as a DaemonSet on each AKS node.

## Workbooks and Dashboards

### Azure Monitor Workbooks

**Workbooks** are interactive reports that combine text, KQL queries, metrics, and parameters into a single document. They are more powerful than dashboards for detailed analysis:
- Run KQL queries inline
- Add parameter pickers (time range, subscription, resource)
- Drill down from summaries to details
- Share as templates across teams

### Azure Dashboards

**Dashboards** are pinned panels — metrics charts, KQL query results, resource health tiles — displayed side-by-side. Simpler than workbooks; designed for at-a-glance operational views.

### Grafana integration

**Azure Managed Grafana** is a fully managed Grafana service with built-in Azure Monitor and Prometheus data sources — teams already familiar with Grafana can use it instead of Azure native dashboards.

## Working with Azure Monitor via Python

In [ ]:
# pip install azure-identity azure-monitor-query azure-mgmt-monitor azure-mgmt-loganalytics

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, MetricsQueryClient, MetricAggregationType
from azure.mgmt.monitor import MonitorManagementClient
from azure.mgmt.monitor.models import (
    MetricAlertResource, MetricAlertSingleResourceMultipleMetricCriteria,
    MetricCriteria, ActionGroup, ActionGroupResource
)
from datetime import datetime, timezone, timedelta

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"

logs_client    = LogsQueryClient(credential)
metrics_client = MetricsQueryClient(credential)
monitor        = MonitorManagementClient(credential, subscription_id)

In [ ]:
from azure.monitor.query import LogsQueryStatus

workspace_id = "<log-analytics-workspace-id>"

# KQL query: top 10 failing requests in last hour
kql = """
requests
| where timestamp > ago(1h)
| where success == false
| summarize FailCount = count() by name, resultCode
| order by FailCount desc
| top 10 by FailCount
"""

result = logs_client.query_workspace(
    workspace_id=workspace_id,
    query=kql,
    timespan=timedelta(hours=1)
)

if result.status == LogsQueryStatus.SUCCESS:
    table = result.tables[0]
    print(f"Columns: {[c.name for c in table.columns]}")
    for row in table.rows:
        print(f"  {row[0]:<40} {row[1]:<10} failures: {row[2]}")

In [ ]:
# Query CPU percentage for a VM
vm_resource_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.Compute/virtualMachines/my-vm"
)

end   = datetime.now(timezone.utc)
start = end - timedelta(hours=1)

result = metrics_client.query_resource(
    resource_uri=vm_resource_id,
    metric_names=["Percentage CPU"],
    timespan=(start, end),
    granularity=timedelta(minutes=5),
    aggregations=[MetricAggregationType.AVERAGE, MetricAggregationType.MAXIMUM]
)

for metric in result.metrics:
    print(f"\nMetric: {metric.name}")
    for ts in metric.timeseries:
        for point in ts.data:
            if point.average is not None:
                print(f"  {point.timestamp}  avg={point.average:.1f}%  max={point.maximum:.1f}%")

In [ ]:
from azure.mgmt.monitor.models import EmailReceiver

# Create an action group with email notification
ag = monitor.action_groups.create_or_update(
    resource_group, "ops-alerts-ag",
    ActionGroupResource(
        location="global",
        group_short_name="OpsAlerts",
        enabled=True,
        email_receivers=[
            EmailReceiver(
                name="ops-team",
                email_address="ops@contoso.com",
                use_common_alert_schema=True
            )
        ]
    )
)
print(f"Action group: {ag.name}  id: {ag.id}")

In [ ]:
# Create a metric alert: fire when CPU > 85% average over 5 minutes
alert = monitor.metric_alerts.create_or_update(
    resource_group, "high-cpu-alert",
    MetricAlertResource(
        location="global",
        description="Alert when VM CPU exceeds 85%",
        severity=2,                    # Warning
        enabled=True,
        evaluation_frequency="PT1M",   # check every 1 minute
        window_size="PT5M",            # aggregate over 5 minutes
        scopes=[vm_resource_id],
        criteria=MetricAlertSingleResourceMultipleMetricCriteria(
            all_of=[
                MetricCriteria(
                    name="HighCPU",
                    metric_name="Percentage CPU",
                    metric_namespace="Microsoft.Compute/virtualMachines",
                    operator="GreaterThan",
                    threshold=85.0,
                    time_aggregation="Average",
                    criterion_type="StaticThresholdCriterion"
                )
            ]
        ),
        actions=[{"action_group_id": ag.id}]
    )
)
print(f"Alert rule created: {alert.name}  severity: {alert.severity}")

In [ ]:
# List fired alert instances in the last 24 hours
from azure.mgmt.monitor import MonitorManagementClient

for alert_instance in monitor.alerts.get_all(
    time_range="1d",
    severity="Sev2"
):
    props = alert_instance.properties
    print(f"  [{props.severity}] {props.alert_rule:<40} state: {props.alert_state}  "
          f"fired: {props.start_date_time}")

In [ ]:
from azure.mgmt.monitor.models import DiagnosticSettingsResource, LogSettings, MetricSettings, RetentionPolicy

workspace_resource_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.OperationalInsights/workspaces/my-workspace"
)

storage_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.Storage/storageAccounts/mystorageaccount"
)

# Enable diagnostic settings on a Key Vault — send audit logs to Log Analytics
kv_resource_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.KeyVault/vaults/my-keyvault"
)

monitor.diagnostic_settings.create_or_update(
    resource_uri=kv_resource_id,
    name="kv-diagnostics",
    parameters=DiagnosticSettingsResource(
        workspace_id=workspace_resource_id,
        logs=[
            LogSettings(category="AuditEvent", enabled=True,
                        retention_policy=RetentionPolicy(enabled=False, days=0))
        ],
        metrics=[
            MetricSettings(category="AllMetrics", enabled=True,
                           retention_policy=RetentionPolicy(enabled=False, days=0))
        ]
    )
)
print("Diagnostic settings enabled for Key Vault")

## Summary

| Concept | Key Takeaway |
|---|---|
| Azure Monitor | Unified observability — collects metrics, logs, traces from all Azure resources |
| Metrics | Numerical time-series; 93-day retention; near-real-time; free for platform metrics |
| Metrics Explorer | Plot, aggregate, split by dimension; pin to dashboard |
| Diagnostic Settings | Route resource logs + metrics to Log Analytics / Storage / Event Hub |
| Activity Log | Control-plane operations — who changed what; 90-day default retention |
| Resource Logs | Data-plane operations — not collected by default; enable via Diagnostic Settings |
| Log Analytics workspace | Central log store; query with KQL; 30–730 days interactive retention |
| KQL | Pipe-based query language — where, project, summarize, join, render |
| Application Insights | APM — requests, dependencies, exceptions, traces, availability tests |
| Workspace-based App Insights | Stores data in Log Analytics — cross-resource KQL queries |
| Distributed tracing | Correlation ID across services; Application Map for visualisation |
| Metric alert | Fires when a metric crosses a static or dynamic threshold |
| Log alert | Fires when a KQL query returns results meeting a condition |
| Activity log alert | Fires on control-plane events (VM deleted, policy non-compliant) |
| Dynamic thresholds | ML-based baseline — reduces false positives vs static thresholds |
| Action group | Reusable set of notifications/actions (email, webhook, Function, runbook) |
| Alert processing rule | Suppress alerts during maintenance; add action groups by scope |
| Azure Monitor Agent | Replaces legacy agents; configured via Data Collection Rules |
| VM Insights | Out-of-the-box performance + dependency map for VMs |
| Container Insights | AKS monitoring — pod logs, node metrics, container performance |
| Workbooks | Interactive reports combining KQL, metrics, and text |
| Azure Managed Grafana | Managed Grafana with built-in Azure Monitor + Prometheus data sources |